<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/Adult_Utility_Fairness_Tradeoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    demographic_parity_ratio,
    equalized_odds_ratio,
)
import warnings
warnings.filterwarnings('ignore')


# ==========================================
# HELPER FUNCTIONS FOR METRIC NAMING
# ==========================================

def get_metric_full_name(metric: str) -> str:
    """Get full descriptive name for metric."""
    mapping = {
        "accuracy": "Accuracy",
        "f1": "F1-score",
        "dpd": "Demographic Parity Difference",
        "dpr": "Demographic Parity Ratio",
        "eod": "Equalized Odds Difference",
        "eor": "Equalized Odds Ratio",
    }
    return mapping.get(metric, metric.upper())


def get_metric_axis_label(metric: str) -> str:
    """Get appropriate label for metric on plot axis."""
    mapping = {
        "accuracy": "Accuracy",
        "f1": "F1-score",
        "dpd": "DPD (Lower is Better)",
        "dpr": "DPR (Closer to 1 is Better)",
        "eod": "EOD (Lower is Better)",
        "eor": "EOR (Closer to 1 is Better)",
    }
    return mapping.get(metric, metric.upper())


def get_metric_short_interpretation(metric: str) -> str:
    """Get short interpretation of metric direction."""
    if metric in ["dpd", "eod"]:
        return "Lower is better"
    if metric in ["dpr", "eor"]:
        return "Closer to 1 is better"
    if metric in ["accuracy", "f1"]:
        return "Higher is better"
    return ""


# ==========================================
# DATA LOADING AND PREPROCESSING
# ==========================================

def load_and_preprocess_adult() -> pd.DataFrame:
    """
    Load and preprocess UCI Adult dataset.

    Returns:
        df: Preprocessed DataFrame with binary income target
    """
    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "income",
    ]

    dataset_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

    print("Loading UCI Adult dataset...")
    df = pd.read_csv(
        dataset_url,
        names=columns,
        skipinitialspace=True,
        na_values=["?"],
    )

    print(f"  Original shape: {df.shape}")

    # Drop rows with missing values
    df = df.dropna().copy()
    print(f"  After dropping NaN: {df.shape}")

    # Drop fnlwgt (sampling weight, not a causal feature)
    df = df.drop(columns=["fnlwgt", "education", "native-country"])
    print(f"  After dropping weight/location features: {df.shape}")

    # Binarize target
    df["income"] = (df["income"].str.strip() == ">50K").astype(int)

    return df


# ==========================================
# MODEL PIPELINE
# ==========================================

def build_model_pipeline_adult(
    feature_columns: list[str],
    continuous_columns: list[str],
    model_type: str = "logistic",
) -> Pipeline:
    """Build a preprocessing + ML model pipeline for Adult data."""
    cont_cols = [c for c in feature_columns if c in continuous_columns]
    cat_cols = [c for c in feature_columns if c not in continuous_columns]

    transformers = []

    if cont_cols:
        transformers.append(
            (
                "cont",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                cont_cols,
            )
        )

    if cat_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=50)),
                    ]
                ),
                cat_cols,
            )
        )

    preprocess = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    if model_type == "logistic":
        model = LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            random_state=42,
            n_jobs=-1,
            tol=0.01,
        )
    elif model_type == "random_forest":
        model = RandomForestClassifier(
            n_estimators=50,
            max_depth=10,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=10,
        )
    elif model_type == "gradient_boosting":
        model = GradientBoostingClassifier(
            n_estimators=50,
            max_depth=4,
            learning_rate=0.05,
            random_state=42,
            subsample=0.8,
            min_samples_leaf=5,
            verbose=0,
        )
    else:
        raise ValueError(
            f"Unknown model_type: {model_type}. "
            "Choose from 'logistic', 'random_forest', 'gradient_boosting'"
        )

    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model),
        ]
    )


# ==========================================
# EVALUATION
# ==========================================

def evaluate_feature_sets_adult(
    data: pd.DataFrame,
    target: str,
    sensitive_attr: str,
    mb_features: list[str],
    proxy_features: list[str],
    continuous_columns: list[str],
    model_types: list[str] = ["logistic", "random_forest", "gradient_boosting"],
    n_splits: int = 30,
    test_size: float = 0.30,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Repeated stratified holdout evaluation."""

    mb_features_set = set(mb_features) - {sensitive_attr}
    proxy_features_set = set(proxy_features)

    fs_mb = sorted(mb_features_set | {sensitive_attr})
    fs_fair = sorted(mb_features_set - proxy_features_set)

    print(f"\n[Feature Sets Summary]")
    print(f"  MB_Shield: {len(fs_mb)} features (MB + sensitive attribute)")
    print(f"    Features: {fs_mb}")
    print(f"  Fair_Filter: {len(fs_fair)} features (MB - proxy features)")
    print(f"    Features: {fs_fair}")
    print(f"  Removed proxy features: {sorted(proxy_features_set)}")
    print(f"  Features difference: {len(fs_mb) - len(fs_fair)}")

    feature_sets = {
        "MB_Shield": fs_mb,
        "Fair_Filter": fs_fair,
    }

    fairness_metrics = {
        "dpd": demographic_parity_difference,
        "dpr": demographic_parity_ratio,
        "eod": equalized_odds_difference,
        "eor": equalized_odds_ratio,
    }

    X = data.drop(columns=[target, sensitive_attr])
    y = data[target].to_numpy()
    s = data[sensitive_attr].to_numpy()

    # Ensure y is integer
    if y.dtype == bool:
        y = y.astype(int)

    # Ensure s is integer
    if not np.issubdtype(s.dtype, np.integer):
        # Label-encode the sensitive attribute if it's categorical
        le = LabelEncoder()
        s = le.fit_transform(s.astype(str))

    print(f"\n[DEBUG] Data shapes: X={X.shape}, y={y.shape}, s={s.shape}")
    print(f"[DEBUG] y dtype: {y.dtype}, unique values: {np.unique(y)}")
    print(f"[DEBUG] s dtype: {s.dtype}, unique values: {np.unique(s)}")

    splitter = StratifiedShuffleSplit(
        n_splits=n_splits,
        test_size=test_size,
        random_state=random_state,
    )

    rows = []
    total_iterations = n_splits * len(feature_sets) * len(model_types)
    iteration = 0
    errors_count = 0
    success_count = 0

    for split_idx, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        print(f"\n[Split {split_idx}/{n_splits}]")

        X_train_full = X.iloc[train_idx].copy()
        X_test_full = X.iloc[test_idx].copy()
        y_train = y[train_idx]
        y_test = y[test_idx]
        s_test = s[test_idx]

        for config_name, feature_cols in feature_sets.items():
            feature_cols_valid = [f for f in feature_cols if f in X.columns]

            if not feature_cols_valid:
                print(f"\n  [WARNING] No valid features for {config_name}")
                continue

            for model_type in model_types:
                iteration += 1
                print(f"  {config_name:15} + {model_type:20} [{iteration}/{total_iterations}]", end=' ', flush=True)

                try:
                    pipe = build_model_pipeline_adult(feature_cols_valid, continuous_columns, model_type)

                    X_train = X_train_full[feature_cols_valid]
                    X_test = X_test_full[feature_cols_valid]

                    # FIXED: Check for NaN values that work with both numeric and categorical
                    train_has_nan = False
                    test_has_nan = False

                    for col in X_train.columns:
                        if X_train[col].isna().any():
                            train_has_nan = True
                            break

                    for col in X_test.columns:
                        if X_test[col].isna().any():
                            test_has_nan = True
                            break

                    if train_has_nan or test_has_nan:
                        print(f"✗ NaN in features")
                        errors_count += 1
                        continue

                    try:
                        pipe.fit(X_train, y_train)
                    except Exception as fit_error:
                        print(f"✗ Fit: {str(fit_error)[:30]}")
                        errors_count += 1
                        continue

                    try:
                        y_pred = pipe.predict(X_test)
                    except Exception as pred_error:
                        print(f"✗ Predict: {str(pred_error)[:30]}")
                        errors_count += 1
                        continue

                    # Validate predictions
                    if y_pred is None or len(y_pred) == 0:
                        print(f"✗ Empty predictions")
                        errors_count += 1
                        continue

                    # Convert to int if needed
                    if y_pred.dtype == bool:
                        y_pred = y_pred.astype(int)
                    elif not np.issubdtype(y_pred.dtype, np.integer):
                        y_pred = y_pred.astype(int)

                    # Compute metrics
                    acc = accuracy_score(y_test, y_pred)
                    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

                    row = {
                        "split": split_idx,
                        "config": config_name,
                        "model": model_type,
                        "n_features": len(feature_cols_valid),
                        "features_used": tuple(feature_cols_valid),
                        "accuracy": acc,
                        "f1": f1,
                    }

                    # Calculate fairness metrics
                    metric_count = 0
                    for metric_name, metric_func in fairness_metrics.items():
                        try:
                            metric_value = metric_func(
                                y_true=y_test.astype(int),
                                y_pred=y_pred.astype(int),
                                sensitive_features=s_test.astype(int),
                            )

                            if metric_value is not None and not np.isnan(metric_value):
                                row[metric_name] = float(metric_value)
                                metric_count += 1
                            else:
                                row[metric_name] = np.nan

                        except Exception as metric_error:
                            row[metric_name] = np.nan

                    # Append if we got most metrics
                    if metric_count >= 3:
                        rows.append(row)
                        success_count += 1
                        print("✓")
                    else:
                        print(f"✗ Only {metric_count} metrics computed")
                        errors_count += 1

                except Exception as e:
                    print(f"✗ Exception: {str(e)[:30]}")
                    errors_count += 1
                    continue

    print(f"\n[DEBUG] Results: {success_count} successes, {errors_count} errors")

    results = pd.DataFrame(rows)

    if results.empty:
        raise ValueError("No results generated. Check configuration.")

    print(f"✓ Results generated: {len(results)} rows")

    # Generate summary statistics
    summaries = []

    for metric_name in ["accuracy", "f1", "dpd", "dpr", "eod", "eor"]:
        for model_type in model_types:
            subset = results[results["model"] == model_type]

            for config in ["MB_Shield", "Fair_Filter"]:
                config_subset = subset[subset["config"] == config]

                if config_subset.empty or metric_name not in config_subset.columns:
                    continue

                values = config_subset[metric_name].dropna().values

                if len(values) > 0:
                    summaries.append(
                        {
                            "metric": metric_name,
                            "model": model_type,
                            "config": config,
                            "mean": np.mean(values),
                            "std": np.std(values, ddof=1) if len(values) > 1 else 0,
                            "p_value": np.nan,
                        }
                    )

            # Compute differences
            mb_data = subset[subset["config"] == "MB_Shield"]
            fair_data = subset[subset["config"] == "Fair_Filter"]

            if not mb_data.empty and not fair_data.empty:
                mb_vals = mb_data[metric_name].dropna().values
                fair_vals = fair_data[metric_name].dropna().values

                if len(mb_vals) > 1 and len(fair_vals) > 1 and len(mb_vals) == len(fair_vals):
                    diff = fair_vals - mb_vals
                    t_stat, p_val = ttest_rel(fair_vals, mb_vals)

                    summaries.append(
                        {
                            "metric": metric_name,
                            "model": model_type,
                            "config": "Difference (Fair_Filter - MB_Shield)",
                            "mean": np.mean(diff),
                            "std": np.std(diff, ddof=1),
                            "p_value": p_val,
                        }
                    )

    summary_df = pd.DataFrame(summaries)

    return summary_df, results


# ==========================================
# PLOTTING FUNCTIONS
# ==========================================

def plot_fairness_utility_tradeoff_per_metric(
    summary_df: pd.DataFrame,
    metric: str,
    model_types: list[str],
    save_path: str = None,
) -> None:
    """
    Plot accuracy (left) and fairness metric (right) for each model.
    """
    if metric == "accuracy" or metric == "f1":
        # For utility metrics, just plot the metric
        save_path = save_path or f"adult_fairness_utility_tradeoff_{metric}.png"
        fig, ax = plt.subplots(figsize=(10, 5))

        subset = summary_df[
            (summary_df["metric"] == metric)
            & (summary_df["config"].isin(["MB_Shield", "Fair_Filter"]))
        ]

        if subset.empty:
            print(f"Warning: No data for metric {metric}")
            return

        x = np.arange(len(model_types))
        width = 0.35

        mb_data = subset[subset["config"] == "MB_Shield"]
        mb_means = []
        mb_stds = []
        for model in model_types:
            row = mb_data[mb_data["model"] == model]
            if not row.empty:
                mb_means.append(row["mean"].values[0])
                mb_stds.append(row["std"].values[0])
            else:
                mb_means.append(0)
                mb_stds.append(0)

        fair_data = subset[subset["config"] == "Fair_Filter"]
        fair_means = []
        fair_stds = []
        for model in model_types:
            row = fair_data[fair_data["model"] == model]
            if not row.empty:
                fair_means.append(row["mean"].values[0])
                fair_stds.append(row["std"].values[0])
            else:
                fair_means.append(0)
                fair_stds.append(0)

        ax.bar(x - width/2, mb_means, width, yerr=mb_stds, label='MB_Shield',
               capsize=5, alpha=0.8, color='#1f77b4')
        ax.bar(x + width/2, fair_means, width, yerr=fair_stds, label='Fair_Filter',
               capsize=5, alpha=0.8, color='#ff7f0e')

        ax.set_ylabel(get_metric_axis_label(metric), fontweight='bold', fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels([m.replace('_', ' ').title() for m in model_types])
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')

        ax.set_title(f"{get_metric_full_name(metric)} Across Models", fontweight='bold', fontsize=12)

        fig.suptitle(f"Adult Predictive Performance Across Models",
                     fontsize=14, fontweight="bold", y=1.00)
        fig.tight_layout()
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Saved: {save_path}")
        plt.close()
        return

    # For fairness metrics: left=accuracy, right=fairness metric
    save_path = save_path or f"adult_fairness_utility_tradeoff_{metric}.png"
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # LEFT PANEL: ACCURACY
    acc_subset = summary_df[
        (summary_df["metric"] == "accuracy")
        & (summary_df["config"].isin(["MB_Shield", "Fair_Filter"]))
    ]

    if acc_subset.empty:
        print(f"Warning: No accuracy data")
        return

    x = np.arange(len(model_types))
    width = 0.35

    mb_acc_data = acc_subset[acc_subset["config"] == "MB_Shield"]
    mb_acc_means = []
    mb_acc_stds = []
    for model in model_types:
        row = mb_acc_data[mb_acc_data["model"] == model]
        if not row.empty:
            mb_acc_means.append(row["mean"].values[0])
            mb_acc_stds.append(row["std"].values[0])
        else:
            mb_acc_means.append(0)
            mb_acc_stds.append(0)

    fair_acc_data = acc_subset[acc_subset["config"] == "Fair_Filter"]
    fair_acc_means = []
    fair_acc_stds = []
    for model in model_types:
        row = fair_acc_data[fair_acc_data["model"] == model]
        if not row.empty:
            fair_acc_means.append(row["mean"].values[0])
            fair_acc_stds.append(row["std"].values[0])
        else:
            fair_acc_means.append(0)
            fair_acc_stds.append(0)

    axes[0].bar(x - width/2, mb_acc_means, width, yerr=mb_acc_stds, label='MB_Shield',
               capsize=5, alpha=0.8, color='#1f77b4')
    axes[0].bar(x + width/2, fair_acc_means, width, yerr=fair_acc_stds, label='Fair_Filter',
               capsize=5, alpha=0.8, color='#ff7f0e')

    axes[0].set_ylabel("Accuracy", fontweight='bold', fontsize=12)
    axes[0].set_title("Predictive Performance Across Models", fontweight='bold', fontsize=12)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([m.replace('_', ' ').title() for m in model_types])
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')

    # RIGHT PANEL: FAIRNESS METRIC
    metric_subset = summary_df[
        (summary_df["metric"] == metric)
        & (summary_df["config"].isin(["MB_Shield", "Fair_Filter"]))
    ]

    if metric_subset.empty:
        print(f"Warning: No data for metric {metric}")
        return

    mb_fair_data = metric_subset[metric_subset["config"] == "MB_Shield"]
    mb_fair_means = []
    mb_fair_stds = []
    for model in model_types:
        row = mb_fair_data[mb_fair_data["model"] == model]
        if not row.empty:
            mb_fair_means.append(row["mean"].values[0])
            mb_fair_stds.append(row["std"].values[0])
        else:
            mb_fair_means.append(0)
            mb_fair_stds.append(0)

    fair_fair_data = metric_subset[metric_subset["config"] == "Fair_Filter"]
    fair_fair_means = []
    fair_fair_stds = []
    for model in model_types:
        row = fair_fair_data[fair_fair_data["model"] == model]
        if not row.empty:
            fair_fair_means.append(row["mean"].values[0])
            fair_fair_stds.append(row["std"].values[0])
        else:
            fair_fair_means.append(0)
            fair_fair_stds.append(0)

    axes[1].bar(x - width/2, mb_fair_means, width, yerr=mb_fair_stds, label='MB_Shield',
               capsize=5, alpha=0.8, color='#1f77b4')
    axes[1].bar(x + width/2, fair_fair_means, width, yerr=fair_fair_stds, label='Fair_Filter',
               capsize=5, alpha=0.8, color='#ff7f0e')

    axes[1].set_ylabel(get_metric_axis_label(metric), fontweight='bold', fontsize=12)

    # Right panel title based on metric
    if metric == "dpd":
        axes[1].set_title("Demographic Parity Difference Across Models", fontweight='bold', fontsize=12)
    elif metric == "dpr":
        axes[1].set_title("Demographic Parity Ratio Across Models", fontweight='bold', fontsize=12)
    elif metric == "eod":
        axes[1].set_title("Equalized Odds Difference Across Models", fontweight='bold', fontsize=12)
    elif metric == "eor":
        axes[1].set_title("Equalized Odds Ratio Across Models", fontweight='bold', fontsize=12)

    axes[1].set_xticks(x)
    axes[1].set_xticklabels([m.replace('_', ' ').title() for m in model_types])
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3, axis='y')

    if metric in ["dpd", "eod"]:
        axes[1].axhline(0, linewidth=1, linestyle="--", color="gray", alpha=0.7)
    elif metric in ["dpr", "eor"]:
        axes[1].axhline(1, linewidth=1, linestyle="--", color="gray", alpha=0.7)

    fig.suptitle(f"Adult Fairness-Utility Trade-off Across Models ({metric.upper()})",
                 fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {save_path}")
    plt.close()


def plot_per_split_differences_per_metric(
    results_df: pd.DataFrame,
    metric: str,
    model_types: list[str],
    save_path: str = None,
) -> None:
    """Plot per-split differences for accuracy and fairness metric."""
    if save_path is None:
        save_path = f"adult_per_split_differences_{metric}.png"

    n_models = len(model_types)
    fig, axes = plt.subplots(n_models, 2, figsize=(12, 4 * n_models))

    if n_models == 1:
        axes = axes.reshape(1, 2)

    for model_idx, model_type in enumerate(model_types):
        model_data = results_df[results_df["model"] == model_type]

        mb_data = model_data[model_data["config"] == "MB_Shield"]
        fair_data = model_data[model_data["config"] == "Fair_Filter"]

        if mb_data.empty or fair_data.empty:
            continue

        mb_acc = mb_data.pivot_table(index="split", values="accuracy", aggfunc="first")
        fair_acc = fair_data.pivot_table(index="split", values="accuracy", aggfunc="first")

        mb_metric = mb_data.pivot_table(index="split", values=metric, aggfunc="first")
        fair_metric = fair_data.pivot_table(index="split", values=metric, aggfunc="first")

        common_splits = sorted(mb_acc.index.intersection(fair_acc.index))

        if len(common_splits) > 0:
            acc_diff = fair_acc.loc[common_splits, "accuracy"].values - mb_acc.loc[common_splits, "accuracy"].values
            metric_diff = fair_metric.loc[common_splits, metric].values - mb_metric.loc[common_splits, metric].values

            # Accuracy difference subplot
            axes[model_idx, 0].plot(common_splits, acc_diff, marker='o', linestyle='-',
                                   linewidth=2, markersize=6, color='#1f77b4')
            axes[model_idx, 0].axhline(0, linewidth=2, linestyle='-', color='red', alpha=0.7)
            axes[model_idx, 0].set_ylabel("Accuracy Difference", fontweight='bold', fontsize=10)
            axes[model_idx, 0].set_title(f"Accuracy Change - {model_type.replace('_', ' ').title()} (Fair_Filter - MB_Shield)",
                                        fontweight='bold', fontsize=11)
            axes[model_idx, 0].grid(True, alpha=0.3)

            # Metric difference subplot
            axes[model_idx, 1].plot(common_splits, metric_diff, marker='o', linestyle='-',
                                   linewidth=2, markersize=6, color='#1f77b4')

            if metric in ["dpd", "eod"]:
                axes[model_idx, 1].axhline(0, linewidth=2, linestyle='-', color='red', alpha=0.7)
            elif metric in ["dpr", "eor"]:
                axes[model_idx, 1].axhline(0, linewidth=2, linestyle='-', color='red', alpha=0.7)

            axes[model_idx, 1].set_ylabel(f"{get_metric_axis_label(metric)} Difference", fontweight='bold', fontsize=10)
            axes[model_idx, 1].set_title(f"{get_metric_full_name(metric)} Change - {model_type.replace('_', ' ').title()}\n(Fair_Filter - MB_Shield)",
                                        fontweight='bold', fontsize=10)
            axes[model_idx, 1].grid(True, alpha=0.3)

            # Add interpretation note
            if metric in ["dpd", "eod"]:
                interp_text = "Negative = lower disparity\nunder Fair_Filter"
            elif metric in ["dpr", "eor"]:
                interp_text = "Positive = movement toward\nbetter parity (ratio → 1)"
            else:
                interp_text = ""

            if interp_text:
                axes[model_idx, 1].text(0.02, 0.98, interp_text,
                                       transform=axes[model_idx, 1].transAxes,
                                       fontsize=8, verticalalignment='top',
                                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

    axes[-1, 0].set_xlabel("Split", fontweight='bold', fontsize=10)
    axes[-1, 1].set_xlabel("Split", fontweight='bold', fontsize=10)

    fig.suptitle(f"Adult Per-Split Differences Across Models ({metric.upper()})",
                 fontsize=14, fontweight="bold", y=0.995)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {save_path}")
    plt.close()


def plot_accuracy_fairness_tradeoff_analysis(
    summary_df: pd.DataFrame,
    metric: str,
    model_types: list[str],
    save_path: str = None,
) -> None:
    """Plot accuracy vs fairness metric scatter with error bars."""
    if save_path is None:
        save_path = f"adult_accuracy_fairness_tradeoff_{metric}.png"

    fig, ax = plt.subplots(figsize=(10, 7))

    colors_model = ['#1f77b4', '#ff7f0e', '#2ca02c']
    markers_config = {'MB_Shield': 'o', 'Fair_Filter': 's'}

    accuracy_subset = summary_df[
        (summary_df["metric"] == "accuracy")
        & (summary_df["config"].isin(["MB_Shield", "Fair_Filter"]))
    ]

    metric_subset = summary_df[
        (summary_df["metric"] == metric)
        & (summary_df["config"].isin(["MB_Shield", "Fair_Filter"]))
    ]

    for model_idx, model_type in enumerate(model_types):
        for config_idx, config in enumerate(['MB_Shield', 'Fair_Filter']):
            acc_row = accuracy_subset[
                (accuracy_subset["model"] == model_type) &
                (accuracy_subset["config"] == config)
            ]
            metric_row = metric_subset[
                (metric_subset["model"] == model_type) &
                (metric_subset["config"] == config)
            ]

            if not acc_row.empty and not metric_row.empty:
                acc_mean = acc_row["mean"].values[0]
                acc_std = acc_row["std"].values[0]
                metric_mean = metric_row["mean"].values[0]
                metric_std = metric_row["std"].values[0]

                marker = markers_config[config]
                label = f"{config} - {model_type.replace('_', ' ').title()}"

                ax.errorbar(metric_mean, acc_mean,
                          xerr=metric_std, yerr=acc_std,
                          fmt=marker, markersize=12, linewidth=2, capsize=5,
                          color=colors_model[model_idx], alpha=0.7,
                          label=label)

    # Metric-specific x-axis label
    if metric in ["dpd", "eod"]:
        x_label = f"Fairness ({metric.upper()}) - Lower is Better"
    elif metric in ["dpr", "eor"]:
        x_label = f"Fairness ({metric.upper()}) - Closer to 1 is Better"
    else:
        x_label = f"Fairness ({metric.upper()})"

    ax.set_xlabel(x_label, fontweight='bold', fontsize=12)
    ax.set_ylabel("Accuracy - Higher is Better", fontweight='bold', fontsize=12)
    ax.set_title(f"Adult Accuracy-Fairness Trade-off Analysis ({metric.upper()})",
                fontweight='bold', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='best')

    # Add reference line
    if metric in ["dpd", "eod"]:
        ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    elif metric in ["dpr", "eor"]:
        ax.axvline(1, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {save_path}")
    plt.close()


# ==========================================
# REPORTING
# ==========================================

def create_summary_report(
    df: pd.DataFrame,
    mb_features: list[str],
    proxy_features: list[str],
    summary_df: pd.DataFrame,
    split_results_df: pd.DataFrame,
    save_path: str = "adult_fairness_utility_report.txt"
) -> None:
    """Generate comprehensive text report."""

    with open(save_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("ADULT FAIRNESS-UTILITY ANALYSIS REPORT\n")
        f.write("Configured Markov Blanket and Proxy Filtering Audit\n")
        f.write("="*80 + "\n\n")

        f.write("DATASET SUMMARY:\n")
        f.write(f"  Total rows (after preprocessing): {len(df)}\n")
        f.write(f"  Total columns: {len(df.columns)}\n")
        f.write(f"\n  Target: income (binary, 1 = >50K)\n")
        f.write(f"    Distribution: {dict(df['income'].value_counts())}\n")
        f.write(f"\n  Sensitive Attribute: sex\n")
        f.write(f"    Distribution: {dict(df['sex'].value_counts())}\n\n")

        f.write("="*80 + "\n")
        f.write("FEATURE CONFIGURATION (Configured, Not Learned):\n")
        f.write("="*80 + "\n\n")

        f.write(f"Markov Blanket Features (MB): {len(mb_features)}\n")
        for i, feat in enumerate(mb_features, 1):
            f.write(f"  {i:2d}. {feat}\n")

        f.write(f"\nProxy Features (to be removed): {len(proxy_features)}\n")
        for i, feat in enumerate(proxy_features, 1):
            f.write(f"  {i:2d}. {feat}\n")

        f.write(f"\nMB_Shield (MB + sex):\n")
        f.write(f"  Total: {len(mb_features) + 1} features\n")

        f.write(f"\nFair_Filter (MB - proxy):\n")
        f.write(f"  Total: {len(mb_features) - len(proxy_features)} features\n")

        f.write(f"\nRemoved proxy features: {len(proxy_features)}\n")
        f.write(f"Expected accuracy cost: Feature removal may reduce predictive power.\n\n")

        f.write("="*80 + "\n")
        f.write("FAIRNESS METRICS:\n")
        f.write("="*80 + "\n\n")

        f.write("DPD = Demographic Parity Difference (Lower is Better)\n")
        f.write("DPR = Demographic Parity Ratio (Closer to 1 is Better)\n")
        f.write("EOD = Equalized Odds Difference (Lower is Better)\n")
        f.write("EOR = Equalized Odds Ratio (Closer to 1 is Better)\n\n")

        f.write("="*80 + "\n")
        f.write("SUMMARY STATISTICS:\n")
        f.write("="*80 + "\n\n")

        # Summary table
        summary_pivot = summary_df[
            (summary_df["config"].isin(["MB_Shield", "Fair_Filter"])) &
            (~summary_df["metric"].str.contains("Difference"))
        ].copy()

        for metric in ["accuracy", "f1", "dpd", "dpr", "eod", "eor"]:
            f.write(f"\n{get_metric_full_name(metric)}:\n")
            f.write("-" * 80 + "\n")

            metric_data = summary_pivot[summary_pivot["metric"] == metric]

            for model in ["logistic", "random_forest", "gradient_boosting"]:
                model_data = metric_data[metric_data["model"] == model]
                f.write(f"\n  {model.upper()}:\n")

                for config in ["MB_Shield", "Fair_Filter"]:
                    config_data = model_data[model_data["config"] == config]
                    if not config_data.empty:
                        mean = config_data["mean"].values[0]
                        std = config_data["std"].values[0]
                        f.write(f"    {config:15} : {mean:.4f} ± {std:.4f}\n")

        # Differences
        f.write("\n" + "="*80 + "\n")
        f.write("DIFFERENCES (Fair_Filter - MB_Shield):\n")
        f.write("="*80 + "\n\n")

        diff_data = summary_df[
            summary_df["config"] == "Difference (Fair_Filter - MB_Shield)"
        ].copy()

        for metric in ["accuracy", "f1", "dpd", "dpr", "eod", "eor"]:
            f.write(f"\n{get_metric_full_name(metric)}:\n")
            f.write("-" * 80 + "\n")

            metric_diff = diff_data[diff_data["metric"] == metric]

            for model in ["logistic", "random_forest", "gradient_boosting"]:
                model_diff = metric_diff[metric_diff["model"] == model]
                if not model_diff.empty:
                    mean_diff = model_diff["mean"].values[0]
                    std_diff = model_diff["std"].values[0]
                    p_val = model_diff["p_value"].values[0]

                    sig_marker = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                    f.write(f"  {model:20} : {mean_diff:+.4f} ± {std_diff:.4f}  (p={p_val:.4f}) {sig_marker}\n")

        f.write("\n" + "="*80 + "\n")
        f.write("KEY FINDINGS AND INTERPRETATION:\n")
        f.write("="*80 + "\n\n")

        f.write("1. ACCURACY IMPACT:\n")
        acc_diff = diff_data[(diff_data["metric"] == "accuracy") & (diff_data["model"] == "random_forest")]
        if not acc_diff.empty:
            f.write(f"   Fair_Filter typically shows accuracy cost (mean difference ~{acc_diff['mean'].values[0]:.4f}).\n")
            f.write(f"   This represents the realized predictive cost of proxy filtering.\n\n")

        f.write("2. FAIRNESS METRIC CHANGES:\n")
        f.write("   DPD/EOD (lower is better):\n")
        f.write("     Negative differences indicate Fair_Filter reduces demographic/outcome disparities.\n\n")
        f.write("   DPR/EOR (closer to 1 is better):\n")
        f.write("     Positive differences indicate Fair_Filter moves ratios closer to parity (toward 1).\n\n")

        f.write("3. TRADE-OFF CHARACTERIZATION:\n")
        f.write("   Results are consistent with the minimality-sufficiency compromise:\n")
        f.write("   - Removing proxy features reduces fairness-relevant dependencies.\n")
        f.write("   - This typically incurs some accuracy cost.\n")
        f.write("   - The trade-off magnitude depends on proxy feature predictive power.\n\n")

        f.write("4. STATISTICAL SIGNIFICANCE:\n")
        f.write("   *** = p < 0.001, ** = p < 0.01, * = p < 0.05, ns = p >= 0.05\n")
        f.write("   Paired t-tests compare Fair_Filter vs MB_Shield across splits.\n\n")

        f.write("="*80 + "\n")
        f.write("IMPORTANT DISCLAIMERS:\n")
        f.write("="*80 + "\n\n")

        f.write("• This audit uses a CONFIGURED feature set (MB and proxies are manually specified).\n")
        f.write("  It is NOT a discovery of the true causal Markov Blanket.\n\n")

        f.write("• Results should be interpreted as an EMPIRICAL AUDIT of the configured\n")
        f.write("  feature sets' behavior, not as proof of a causal graph.\n\n")

        f.write("• The realized fairness-utility trade-off is specific to:\n")
        f.write("  - The dataset and its causal structure.\n")
        f.write("  - The choice of fairness metric.\n")
        f.write("  - The machine learning algorithm.\n")
        f.write("  - The feature set configuration.\n\n")

        f.write("• Proxy identification via mutual information is heuristic.\n")
        f.write("  It may not capture all fairness-relevant dependencies.\n\n")

        f.write("="*80 + "\n")

    print(f"✓ Report saved: {save_path}")


# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print("="*80)
    print("UCI ADULT FAIRNESS-UTILITY ANALYSIS")
    print("Configured Markov Blanket and Proxy Filtering Audit")
    print("="*80)

    # Load and preprocess
    df = load_and_preprocess_adult()
    print(f"\n✓ Dataset preprocessed: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Income distribution: {dict(df['income'].value_counts())}")
    print(f"  Sex distribution: {dict(df['sex'].value_counts())}")

    # Configure feature sets
    target_col = "income"
    sensitive_col = "sex"

    mb_features = [
        "age",
        "education-num",
        "marital-status",
        "occupation",
        "relationship",
        "hours-per-week",
        "capital-gain",
        "capital-loss",
        "workclass",
        "race",
    ]

    proxy_features = [
        "relationship",
        "marital-status",
        "occupation",
    ]

    continuous_columns = [
        "age",
        "education-num",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
    ]

    # Filter to available columns
    mb_features = [f for f in mb_features if f in df.columns]
    proxy_features = [f for f in proxy_features if f in df.columns]
    continuous_columns = [f for f in continuous_columns if f in df.columns]

    print(f"\n[Feature Configuration]")
    print(f"  CONFIGURED (not learned):")
    print(f"    MB Features: {mb_features}")
    print(f"    Proxy Features: {proxy_features}")
    print(f"    Continuous: {continuous_columns}")

    model_types = ["logistic", "random_forest", "gradient_boosting"]

    print(f"\nModels: {model_types}")
    print(f"Fairness Metrics: DPD (Demographic Parity Difference),")
    print(f"                   DPR (Demographic Parity Ratio),")
    print(f"                   EOD (Equalized Odds Difference),")
    print(f"                   EOR (Equalized Odds Ratio)")
    print(f"Splits: 30")
    print(f"\nStarting evaluation...")

    try:
        summary_df, split_results_df = evaluate_feature_sets_adult(
            data=df,
            target=target_col,
            sensitive_attr=sensitive_col,
            mb_features=mb_features,
            proxy_features=proxy_features,
            continuous_columns=continuous_columns,
            model_types=model_types,
            n_splits=30,
            test_size=0.30,
            random_state=42,
        )
    except Exception as e:
        print(f"Error during evaluation: {e}")
        import traceback
        traceback.print_exc()
        raise

    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_rows", None)

    # Save results
    print("\n" + "=" * 80)
    print("=== SAVING RESULTS ===")
    print("=" * 80)

    summary_df.to_csv("adult_fairness_utility_summary.csv", index=False)
    print("✓ Saved: adult_fairness_utility_summary.csv")

    split_results_df.to_csv("adult_fairness_utility_split_results.csv", index=False)
    print("✓ Saved: adult_fairness_utility_split_results.csv")

    # Print summary
    print("\n" + "=" * 80)
    print("=== SUMMARY STATISTICS ===")
    print("=" * 80)
    print(summary_df.round(4).to_string(index=False))

    print("\n" + "=" * 80)
    print("=== FIRST 30 SPLIT-LEVEL RESULTS ===")
    print("=" * 80)
    print(split_results_df.head(30).round(4).to_string(index=False))

    # Generate plots
    print("\n" + "=" * 80)
    print("=== GENERATING PLOTS ===")
    print("=" * 80)

    fairness_metrics = ["dpd", "dpr", "eod", "eor"]

    print("\n[1/3] Generating Fairness-Utility Trade-off Plots...")
    for metric in fairness_metrics:
        try:
            plot_fairness_utility_tradeoff_per_metric(summary_df, metric, model_types)
        except Exception as e:
            print(f"  ✗ {metric} - {str(e)[:50]}")

    print("\n[2/3] Generating Per-Split Differences Plots...")
    for metric in fairness_metrics:
        try:
            plot_per_split_differences_per_metric(split_results_df, metric, model_types)
        except Exception as e:
            print(f"  ✗ {metric} - {str(e)[:50]}")

    print("\n[3/3] Generating Accuracy-Fairness Trade-off Scatter Plots...")
    for metric in fairness_metrics:
        try:
            plot_accuracy_fairness_tradeoff_analysis(summary_df, metric, model_types)
        except Exception as e:
            print(f"  ✗ {metric} - {str(e)[:50]}")

    # Generate report
    print("\n" + "=" * 80)
    print("=== GENERATING REPORT ===")
    print("=" * 80)

    create_summary_report(
        df,
        mb_features,
        proxy_features,
        summary_df,
        split_results_df,
    )

    print("\n" + "=" * 80)
    print("✓ ALL ANALYSES COMPLETED!")
    print("=" * 80)
    print("\nGenerated Files:")
    print("  CSV Results:")
    print("    • adult_fairness_utility_summary.csv")
    print("    • adult_fairness_utility_split_results.csv")
    print("  Fairness-Utility Trade-off Plots (4 files):")
    print("    • adult_fairness_utility_tradeoff_dpd.png")
    print("    • adult_fairness_utility_tradeoff_dpr.png")
    print("    • adult_fairness_utility_tradeoff_eod.png")
    print("    • adult_fairness_utility_tradeoff_eor.png")
    print("  Per-Split Differences Plots (4 files):")
    print("    • adult_per_split_differences_dpd.png")
    print("    • adult_per_split_differences_dpr.png")
    print("    • adult_per_split_differences_eod.png")
    print("    • adult_per_split_differences_eor.png")
    print("  Accuracy-Fairness Trade-off Scatter Plots (4 files):")
    print("    • adult_accuracy_fairness_tradeoff_dpd.png")
    print("    • adult_accuracy_fairness_tradeoff_dpr.png")
    print("    • adult_accuracy_fairness_tradeoff_eod.png")
    print("    • adult_accuracy_fairness_tradeoff_eor.png")
    print("  Report:")
    print("    • adult_fairness_utility_report.txt")
    print("\nTotal: 13 files")
    print("=" * 80)

UCI ADULT FAIRNESS-UTILITY ANALYSIS
Configured Markov Blanket and Proxy Filtering Audit
Loading UCI Adult dataset...
  Original shape: (32561, 15)
  After dropping NaN: (30162, 15)
  After dropping weight/location features: (30162, 12)

✓ Dataset preprocessed: (30162, 12)
  Columns: ['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'income']
  Income distribution: {0: np.int64(22654), 1: np.int64(7508)}
  Sex distribution: {'Male': np.int64(20380), 'Female': np.int64(9782)}

[Feature Configuration]
  CONFIGURED (not learned):
    MB Features: ['age', 'education-num', 'marital-status', 'occupation', 'relationship', 'hours-per-week', 'capital-gain', 'capital-loss', 'workclass', 'race']
    Proxy Features: ['relationship', 'marital-status', 'occupation']
    Continuous: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Models: ['logistic', 'random_forest', 'gradie